## Aggregate top property URLs by source

For each target source JSON file listed in `target_filenames`, this notebook reads the source's registered `url` and queries `https://api.data.niaid.nih.gov/v1/query`, filtering on `includedInDataCatalog.url` matching that url (an exact-phrase match works even though the indexed values are often more specific, e.g. product-page URLs, because the field is text-analyzed).

`infectiousAgent.url`, `species.url`, `healthCondition.url`, and `topicCategory.url` are text fields with fielddata disabled, so Elasticsearch rejects aggregating on them directly (400 `search_phase_execution_exception`). Instead, each property is aggregated on its `.name` field (an `aggs` query, `size=0`) to get the top N term names by dataset count, and each name is then resolved to its `url` via a cached `size=1` lookup.

Results are collected into a long-format table with one row per (source, property, term): columns `file name`, `property`, `property.url`.

In [1]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm


In [2]:
script_path = Path.cwd()
parent_path = script_path.parent
result_path = parent_path / 'nde-metadata-corrections' / 'metadata_for_DDE' / 'dataCatalogs'
#query_url = 'https://api.data.niaid.nih.gov/v1/query'
query_url = 'https://api-staging.data.niaid.nih.gov/v1/query'
request_timeout = 60
top_n_terms = 10

# List of source JSON filenames (as found in result_path) to pull url properties from.
#target_filenames: list[str] = ["ATCC.json", "UCCCB.json"]
target_filenames: list[str] = ["ATCC.json", "BioSample.json", "BioStudies.json", "CCUG.json", 
                              "KCTC.json","Leibniz-Institute DSMZ.json", "National Omics Data Encyclopedia.json","UCCCB.json"]

properties_to_aggregate = ['infectiousAgent', 'species', 'healthCondition', 'topicCategory','measurementTechnique','variableMeasured']

print(f'Using data catalog directory: {result_path}')
print(f'Target files: {target_filenames}')

Using data catalog directory: C:\Users\gtsueng\Anaconda3\envs\nde\nde-metadata-corrections\metadata_for_DDE\dataCatalogs
Target files: ['ATCC.json', 'BioSample.json', 'BioStudies.json', 'CCUG.json', 'KCTC.json', 'Leibniz-Institute DSMZ.json', 'National Omics Data Encyclopedia.json', 'UCCCB.json']


In [3]:
def fetch_query_page(params: dict) -> dict:
    response = requests.get(query_url, params=params, timeout=request_timeout)
    response.raise_for_status()
    return response.json()


def escape_query_value(value: str) -> str:
    return value.replace('\\', '\\\\').replace('"', '\\"')


def normalize_to_list(value):
    if isinstance(value, list):
        return value
    if value in [None, '', [], {}, 'None']:
        return []
    return [value]


def load_target_records(directory: Path, filenames: list[str]) -> dict[str, dict]:
    records = {}
    for filename in filenames:
        file_path = directory / filename
        if not file_path.exists():
            print(f'Warning: {filename} not found in {directory}')
            continue
        records[filename] = json.loads(file_path.read_text(encoding='utf-8'))
    return records

In [4]:
def get_top_term_names_for_source(source_url: str, property_names: list[str], facet_size: int) -> dict[str, list[str]]:
    payload = fetch_query_page(
        params={
            'q': f'includedInDataCatalog.url:"{escape_query_value(source_url)}"',
            'aggs': ','.join(f'{property_name}.name' for property_name in property_names),
            'facet_size': facet_size,
            'size': 0
        }
    )
    facets = payload.get('facets', {})
    return {
        property_name: [
            term['term']
            for term in facets.get(f'{property_name}.name', {}).get('terms', [])
            if term.get('term')
        ]
        for property_name in property_names
    }


def resolve_term_url(property_name: str, term: str) -> str | None:
    payload = fetch_query_page(
        params={
            'q': f'{property_name}.name:"{escape_query_value(term)}"',
            'fields': f'{property_name}.name,{property_name}.url',
            'size': 1
        }
    )
    hits = payload.get('hits', [])
    if not hits:
        return None

    # Some sources (e.g. variableMeasured on BioStudies) store a malformed entry where
    # "name" is a list of strings rather than the expected single string. Skip those
    # entries rather than crashing on entry['name'].lower().
    candidates = [
        entry for entry in normalize_to_list(hits[0].get(property_name))
        if isinstance(entry, dict) and isinstance(entry.get('name'), str) and entry.get('name')
    ]
    match = next((entry for entry in candidates if entry['name'].lower() == term.lower()), None)
    entry = match or (candidates[0] if candidates else None)
    return entry.get('url') if entry else None

In [5]:
target_records = load_target_records(result_path, target_filenames)

rows = []
term_url_cache: dict[tuple[str, str], str | None] = {}

for filename, record in tqdm(target_records.items(), desc='Sources', unit='source'):
    file_stem = Path(filename).stem
    source_url = record.get('url')
    if not source_url:
        print(f'Warning: {filename} has no url property, skipping')
        continue

    top_terms_by_property = get_top_term_names_for_source(source_url, properties_to_aggregate, facet_size=top_n_terms)
    for property_name, terms in top_terms_by_property.items():
        for term in terms:
            cache_key = (property_name, term)
            if cache_key not in term_url_cache:
                term_url_cache[cache_key] = resolve_term_url(property_name, term)
            rows.append({
                'file name': file_stem,
                'property': property_name,
                'property.url': term_url_cache[cache_key]
            })

results_df = pd.DataFrame(rows, columns=['file name', 'property', 'property.url'])
print(f'Built {len(results_df)} rows for {len(target_records)} source(s)')
results_df

Sources:   0%|          | 0/8 [00:00<?, ?source/s]

Built 280 rows for 8 source(s)


,file name,property,property.url
0,ATCC,infectiousAgent,https://www.uniprot.org/taxonomy/336400
1,ATCC,infectiousAgent,https://www.uniprot.org/taxonomy/1902
2,ATCC,infectiousAgent,https://www.uniprot.org/taxonomy/1392
3,ATCC,infectiousAgent,https://www.uniprot.org/taxonomy/562
4,ATCC,infectiousAgent,https://www.uniprot.org/taxonomy/2673
...,...,...,...
275,UCCCB,healthCondition,http://purl.obolibrary.org/obo/MONDO_0006094
276,UCCCB,healthCondition,http://purl.obolibrary.org/obo/NCIT_C186461
277,UCCCB,healthCondition,https://id.nlm.nih.gov/mesh/C565753.html
278,UCCCB,healthCondition,http://purl.obolibrary.org/obo/MONDO_0019287


In [6]:
temp_path = script_path / 'temp'


def export_to_excel(df: pd.DataFrame, filename: str | None = None) -> Path:
    temp_path.mkdir(parents=True, exist_ok=True)
    filename = filename or f'{datetime.now().strftime("%Y-%m-%d_%H%M%S")}_property_urls_by_source.xlsx'
    output_path = temp_path / filename
    df.to_excel(output_path, index=False)
    print(f'Wrote {len(df)} rows to {output_path}')
    return output_path


export_to_excel(results_df)


Wrote 280 rows to C:\Users\gtsueng\Anaconda3\envs\nde\nde_meta_batch_converter\temp\2026-07-23_104829_property_urls_by_source.xlsx


WindowsPath('C:/Users/gtsueng/Anaconda3/envs/nde/nde_meta_batch_converter/temp/2026-07-23_104829_property_urls_by_source.xlsx')